# 14.10 — Forecast evaluation & backtesting

A forecast is only honest when evaluated in the order decisions would have been made, with a metric that matches the real cost of error.

Backtesting is the audit layer for every forecasting model in this part. We compare forecasts only at times when the data would really have been known. Save a copy to Drive to edit.

In [ ]:

import numpy as np
import matplotlib.pyplot as plt

SEED = 1306
rng = np.random.default_rng(SEED)


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def interval_coverage(y_true, lower, upper):
    y_true = np.asarray(y_true, dtype=float)
    lower = np.asarray(lower, dtype=float)
    upper = np.asarray(upper, dtype=float)
    inside = (y_true >= lower) & (y_true <= upper)
    return float(np.mean(inside))


def f13_time_series_ladder():
    t = np.arange(72, dtype=float)
    rungs = []

    y1 = np.full_like(t, 10.0)
    rungs.append({"name": "D1 constant", "t": t, "y": y1, "true": y1, "period": 12})

    y2 = 8.0 + 0.18 * t
    rungs.append({"name": "D2 drift", "t": t, "y": y2, "true": y2, "period": 12})

    noise3 = rng.normal(0.0, 0.45, size=t.size)
    true3 = 8.0 + 0.18 * t
    y3 = true3 + noise3
    rungs.append({"name": "D3 drift + noise", "t": t, "y": y3, "true": true3, "period": 12})

    seasonal4 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true4 = 8.0 + 0.12 * t + seasonal4
    y4 = true4 + rng.normal(0.0, 0.35, size=t.size)
    rungs.append({"name": "D4 seasonal", "t": t, "y": y4, "true": true4, "period": 12})

    seasonal5 = 1.8 * np.sin(2.0 * np.pi * t / 12.0)
    true5 = 8.0 + 0.10 * t + seasonal5
    true5 = true5 + np.where(t >= 48.0, 3.0 + 0.06 * (t - 48.0), 0.0)
    y5 = true5 + rng.normal(0.0, 0.45, size=t.size)
    y5[18] = y5[18] + 5.0
    y5[55] = y5[55] - 4.0
    rungs.append({"name": "D5 outliers + regime shift", "t": t, "y": y5, "true": true5, "period": 12})

    return rungs


def train_test_split_series(rung, train_size=48):
    train = {"t": rung["t"][:train_size], "y": rung["y"][:train_size]}
    test = {"t": rung["t"][train_size:], "y": rung["y"][train_size:], "true": rung["true"][train_size:]}
    return train, test


def print_ladder_preview(rungs):
    for rung in rungs:
        name = rung["name"]
        y = rung["y"]
        sample = np.round(y[:6], 3)
        print(f"{name}: n={y.size}, period={rung['period']}, first six={sample}")


def plot_forecast_panels(results, metric_name="RMSE", marker_key=None, component_key=None):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7))
    axes = axes.ravel()

    for idx, result in enumerate(results):
        ax = axes[idx]
        ax.plot(result["test_t"], result["test_y"], label="observed", color="black")
        ax.plot(result["test_t"], result["forecast"], label="forecast", color="tab:blue")
        ax.plot(result["test_t"], result["test_true"], label="true signal", color="tab:green", alpha=0.7)
        if component_key is not None and component_key in result:
            ax.plot(result["test_t"], result[component_key], label=component_key, color="tab:orange", alpha=0.8)
        if marker_key is not None and marker_key in result:
            marks = result[marker_key]
            for mark in np.atleast_1d(marks):
                ax.axvline(mark, color="tab:red", linestyle="--", alpha=0.6)
        ax.set_title(f"{result['name']}\n{metric_name}={result['metric']:.3f}")
        ax.grid(True, alpha=0.25)

    axes[-1].plot(range(1, len(results) + 1), [r["metric"] for r in results], marker="o")
    axes[-1].set_xticks(range(1, len(results) + 1))
    axes[-1].set_xlabel("D-rung")
    axes[-1].set_ylabel(metric_name)
    axes[-1].set_title(f"{metric_name} curve")
    axes[-1].grid(True, alpha=0.25)
    axes[0].legend(loc="best", fontsize=8)
    fig.tight_layout()



## The concept, built once (D1)

Forecast metrics encode different costs:

$$\operatorname{MAE}=\frac1n\sum_t |y_t-\hat y_t|, \quad \operatorname{RMSE}=\sqrt{\frac1n\sum_t (y_t-\hat y_t)^2}, \quad \operatorname{MAPE}=\frac{100}{n}\sum_t\left|\frac{y_t-\hat y_t}{y_t}\right|$$

The lesson checks are MAE/RMSE $1.000$, MAPE $8.333333\%$, naive scale $2.333333$, MASE $0.428571$, and interval coverage $1.000$.


In [ ]:

def forecast_metrics(y_true, y_pred, train_history=None, lower=None, upper=None):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    errors = y_true - y_pred
    mae = float(np.mean(np.abs(errors)))
    root_mse = float(np.sqrt(np.mean(errors ** 2)))
    mape = float(100.0 * np.mean(np.abs(errors / y_true)))
    out = {"MAE": mae, "RMSE": root_mse, "MAPE": mape}

    if train_history is not None:
        train_history = np.asarray(train_history, dtype=float)
        scale = float(np.mean(np.abs(np.diff(train_history))))
        out["naive_scale"] = scale
        out["MASE"] = mae / scale

    if lower is not None and upper is not None:
        out["coverage"] = interval_coverage(y_true, lower, upper)

    return out

lesson_metrics = forecast_metrics(
    [10.0, 12.0, 15.0],
    [11.0, 13.0, 14.0],
    train_history=[8.0, 10.0, 12.0, 15.0],
    lower=[9.0, 11.0, 14.0],
    upper=[11.0, 13.0, 16.0],
)

assert np.isclose(lesson_metrics["MAE"], 1.000, atol=1e-12)
assert np.isclose(lesson_metrics["RMSE"], 1.000, atol=1e-12)
assert np.isclose(lesson_metrics["MAPE"], 8.333333, atol=5e-7)
assert np.isclose(lesson_metrics["naive_scale"], 2.333333, atol=5e-7)
assert np.isclose(lesson_metrics["MASE"], 0.428571, atol=5e-7)
assert np.isclose(lesson_metrics["coverage"], 1.000, atol=1e-12)
print("lesson checks passed")



Rolling-origin backtesting repeats the decision honestly: fit on the prefix available at an origin, forecast the next horizon, then move forward. This makes leakage visible instead of averaging it away.


In [ ]:

def seasonal_naive_model(history, horizon=1, period=12):
    history = np.asarray(history, dtype=float)
    preds = []
    for step in range(horizon):
        lag = period - step
        if history.size >= lag:
            preds.append(history[-lag])
        else:
            preds.append(history[-1])
    return np.asarray(preds)


def drift_model(history, horizon=1):
    history = np.asarray(history, dtype=float)
    x = np.arange(history.size, dtype=float)
    X = np.column_stack([np.ones_like(x), x])
    beta = np.linalg.lstsq(X, history, rcond=None)[0]
    future_x = np.arange(history.size, history.size + horizon, dtype=float)
    return np.column_stack([np.ones_like(future_x), future_x]) @ beta


def rolling_origin_backtest(y, model, horizon=1, min_train=24, step=3, metric="RMSE", period=12):
    y = np.asarray(y, dtype=float)
    actuals = []
    forecasts = []
    origins = []
    lowers = []
    uppers = []

    for origin in range(min_train, y.size - horizon + 1, step):
        history = y[:origin]
        pred = model(history, horizon=horizon, period=period)
        actual = y[origin:origin + horizon]
        residual_scale = float(np.std(np.diff(history)))
        actuals.extend(actual)
        forecasts.extend(pred)
        origins.extend(np.arange(origin, origin + horizon))
        lowers.extend(pred - 1.2816 * residual_scale)
        uppers.extend(pred + 1.2816 * residual_scale)

    actuals = np.asarray(actuals)
    forecasts = np.asarray(forecasts)
    lowers = np.asarray(lowers)
    uppers = np.asarray(uppers)
    scores = forecast_metrics(actuals, forecasts, train_history=y[:min_train], lower=lowers, upper=uppers)
    return {"actuals": actuals, "forecasts": forecasts, "origins": np.asarray(origins), "scores": scores}



## The dataset ladder

We use the same F13 ladder in every notebook so the method faces increasing time-series difficulty inline: D1 constant, D2 drift, D3 drift plus noise, D4 monthly seasonality, and D5 outliers plus a real regime shift.


In [ ]:

rungs = f13_time_series_ladder()
print_ladder_preview(rungs)


In [ ]:

results = []

for rung in rungs:
    bt = rolling_origin_backtest(rung["y"], seasonal_naive_model, horizon=1, min_train=36, step=3, period=rung["period"])
    metric = bt["scores"]["RMSE"]
    results.append({
        "name": rung["name"],
        "test_t": bt["origins"],
        "test_y": bt["actuals"],
        "test_true": rung["true"][bt["origins"]],
        "forecast": bt["forecasts"],
        "metric": metric,
        "coverage": bt["scores"]["coverage"],
        "folds": bt["origins"],
    })

for result in results:
    print(f"{result['name']}: rolling RMSE={result['metric']:.3f}, coverage={result['coverage']:.3f}")


In [ ]:

plot_forecast_panels(results, metric_name="RMSE", marker_key="folds")



## Pitfall on D5: randomly shuffling time series

A random split leaks future regimes into training. The chronological rolling-origin score is the valid audit because it respects the order decisions would have been made.


In [ ]:

d5 = rungs[-1]
chronological = rolling_origin_backtest(d5["y"], seasonal_naive_model, horizon=1, min_train=36, step=3, period=d5["period"])
perm = rng.permutation(d5["y"].size)
shuffled = d5["y"][perm]
train = shuffled[:48]
test = shuffled[48:]
random_forecast = np.full(test.shape, np.mean(train))
random_rmse = rmse(test, random_forecast)
chrono_rmse = chronological["scores"]["RMSE"]
print(f"random split RMSE={random_rmse:.3f}")
print(f"chronological rolling RMSE={chrono_rmse:.3f}")
print("Only the chronological score answers the forecasting question through the regime shift.")



## Evaluate it + Practice

- Compare chronological rolling-origin RMSE with a naive baseline; report coverage when intervals are present.
- Sanity check: every forecast must be made using only observations before its origin.
- Ablation: shuffle the D5 series before splitting; the apparent error can improve for the wrong reason.
- Failure signal: a method wins only under random splits or only for a horizon unrelated to the decision.
- Track MAE, RMSE, MAPE, MASE, and coverage because each answers a different operational question.

Practice prompts:
1. Change the horizon from 1 to 6 and plot metric growth by horizon.


2. Modify the hardest rung and rerun the same metric table.

3. Write one sentence explaining whether RMSE or coverage is the better decision metric here.